# Part 1 - 실습 3: Amazon Bedrock을 활용한 학교 FAQ 챗봇 구현
**소요시간: 80분** | 난이도: ⭐⭐⭐⭐

## 학습 목표
1. FAQ 데이터를 기반으로 컨텍스트를 주입하는 챗봇을 구현합니다.
2. 대화 히스토리를 관리하는 멀티턴 챗봇 클래스를 설계합니다.
3. 스트리밍 응답으로 실시간 출력을 구현합니다.
4. 챗봇 품질을 평가하는 테스트 케이스를 작성합니다.

## 챗봇 아키텍처
```
사용자 질문
    │
    ▼
FAQ 검색 (키워드 매칭)
    │
    ▼
관련 FAQ → System Prompt 주입
    │
    ▼
Claude converse API
    │
    ▼
스트리밍 응답 출력
```


---
## 🏢 기업 시나리오 — 고객지원 / 사내 헬프데스크 챗봇

당신은 **고객지원(또는 사내 IT 헬프데스크) 챗봇**을 만드는 개발자입니다.
FAQ를 검색해 관련 답변을 모델에 주입(context injection)하고, 멀티턴·스트리밍·품질평가까지 갖춘 실서비스급 챗봇을 구현합니다.

이번 실습에서는 다음을 구현합니다.
1. **FAQ 검색** → 질문에서 키워드를 뽑아 관련 FAQ 찾기
2. **컨텍스트 주입 챗봇** → 검색된 FAQ를 System Prompt에 넣어 정확히 답변
3. **멀티턴 + 스트리밍** → 대화 맥락 유지, 실시간 출력
4. **품질 평가** → 테스트 케이스로 챗봇 정확도 측정

> 💡 이 '검색 후 주입' 방식이 바로 RAG의 단순형입니다. Part 2에서는 키워드 검색을 **벡터 검색(Knowledge Base)** 으로 업그레이드합니다.


In [1]:
# ✅ [제공 코드] 환경 초기화
import boto3, json
from datetime import datetime

bedrock_runtime = boto3.client('bedrock-runtime', region_name='us-east-1')
MODEL_ID = 'us.anthropic.claude-sonnet-4-20250514-v1:0'
print('✅ 환경 초기화 완료')


✅ 환경 초기화 완료


In [2]:
# ✅ [제공 코드] 학교 FAQ 데이터
FAQ_DATA = [
    {
        'category': '수강신청',
        'keywords': ['수강신청', '수업', '강의', '등록', '취소', '시간표'],
        'qa': [
            {'q': '수강신청은 언제 하나요?', 'a': '수강신청은 매 학기 개강 2주 전에 진행됩니다. 1학년은 3월 1일~3일, 2학년 이상은 2월 27~28일입니다.'},
            {'q': '수강신청 취소는 어떻게 하나요?', 'a': '학생포털 → 수강관리 → 수강취소 메뉴에서 가능합니다. 개강 후 2주 이내까지 취소 가능합니다.'},
            {'q': '최대 수강학점은 몇 학점인가요?', 'a': '일반적으로 최대 21학점까지 수강 가능합니다. 직전 학기 성적 우수자(평점 3.5 이상)는 24학점까지 가능합니다.'},
        ]
    },
    {
        'category': '성적',
        'keywords': ['성적', '학점', 'GPA', '이의신청', '재수강', '성적표'],
        'qa': [
            {'q': '성적 이의신청은 어떻게 하나요?', 'a': '성적 발표 후 1주일 이내에 학생포털에서 이의신청 가능합니다. 담당 교수님께 직접 문의하는 방법도 있습니다.'},
            {'q': '재수강 신청 방법은?', 'a': 'C+ 이하 성적 과목을 재수강할 수 있습니다. 수강신청 기간에 동일 과목을 신청하면 됩니다. 재수강 성적은 B+까지만 인정됩니다.'},
            {'q': '졸업 학점은 몇 학점인가요?', 'a': '졸업에는 총 130학점이 필요합니다. 전공 60학점, 교양 30학점, 자유선택 40학점으로 구성됩니다.'},
        ]
    },
    {
        'category': '장학금',
        'keywords': ['장학금', '장학', '등록금', '지원', '신청'],
        'qa': [
            {'q': '성적 장학금 기준은?', 'a': '직전 학기 12학점 이상 수강 및 평점 3.8 이상이면 전액, 3.5~3.79는 반액 장학금을 받을 수 있습니다.'},
            {'q': '근로 장학금 신청은 어떻게 하나요?', 'a': '매 학기 초 학생처 홈페이지에서 신청합니다. 월 40~60시간 근로 조건이며, 시급은 최저임금 기준으로 지급됩니다.'},
        ]
    },
    {
        'category': '학사일정',
        'keywords': ['방학', '개강', '종강', '휴학', '복학', '졸업'],
        'qa': [
            {'q': '2025년 1학기 개강일은?', 'a': '2025년 1학기 개강일은 3월 4일(화)입니다. 종강은 6월 20일이며 기말고사는 6월 10~19일입니다.'},
            {'q': '휴학 신청 방법은?', 'a': '학생포털 → 학적관리 → 휴학신청 메뉴에서 가능합니다. 학기 개강 후 4주 이내에 신청해야 합니다. 군휴학은 연중 가능합니다.'},
        ]
    },
]

print(f'✅ FAQ 데이터 로드 완료: {sum(len(c["qa"]) for c in FAQ_DATA)}개 Q&A')


✅ FAQ 데이터 로드 완료: 10개 Q&A


## ✏️ TODO 1: FAQ 검색 함수 구현

사용자 질문에서 키워드를 추출하여 관련 FAQ를 검색하는 함수를 완성하세요.


In [4]:
# ✏️ TODO 1: FAQ 검색 함수를 완성하세요
def search_faq(query, top_k=3):
    """키워드 기반 FAQ 검색 (관련도 점수 계산)"""
    results = []

    for category in FAQ_DATA:
        # 카테고리 키워드가 쿼리에 포함되면 점수 가산
        category_score = sum(
            1 for kw in category['keywords']    # ← 'keywords'
            if kw in query                 # ← query
        )

        for qa in category['qa']:
            # 질문 키워드와 답변 키워드 매칭 점수
            q_score = sum(1 for word in qa['q'].split() if word in query)
            a_score = sum(1 for word in qa['a'].split() if word in query)
            total_score = category_score * 2 + q_score * 3 + a_score

            if total_score > 0:
                results.append({
                    'category': category['category'],  # ← 'category'
                    'question': qa['q'],
                    'answer'  : qa['a'],        # ← 'a'
                    'score'   : total_score
                })

    # 점수 내림차순 정렬 후 top_k 반환
    results.sort(key=lambda x: x['score'], reverse=True)  # ← 'score'
    return results[:top_k]

# 테스트
test_results = search_faq('수강신청 취소 방법 알려줘')
for r in test_results:
    print(f"[{r['category']}] 점수:{r['score']}  Q: {r['question']}")


[수강신청] 점수:8  Q: 수강신청 취소는 어떻게 하나요?
[수강신청] 점수:5  Q: 최대 수강학점은 몇 학점인가요?
[성적] 점수:5  Q: 재수강 신청 방법은?


## ✏️ TODO 2: FAQ 챗봇 클래스 구현

FAQ 검색 결과를 System Prompt에 주입하고, 대화 히스토리를 관리하는 챗봇 클래스를 완성하세요.


In [5]:
# ✏️ TODO 2: FAQChatbot 클래스를 완성하세요
class FAQChatbot:
    def __init__(self, model_id=MODEL_ID, max_history=10):
        self.model_id     = model_id
        self.max_history  = max_history
        self.history      = []
        self.client       = bedrock_runtime

    def _build_system_prompt(self, faq_results):
        """검색된 FAQ를 System Prompt에 주입"""
        base = """
당신은 대학교 학생처 FAQ 챗봇입니다.
다음 규칙을 반드시 지키세요:
1. 아래 제공된 FAQ 정보를 최우선으로 사용하여 답변하세요.
2. FAQ에 없는 내용은 '해당 내용은 학생처(02-XXX-XXXX)에 문의해 주세요.'라고 안내하세요.
3. 친절하고 공손한 어투를 사용하세요.
4. 답변은 간결하게 3~5문장 이내로 작성하세요.
"""
        if faq_results:
            faq_context = '\n\n[관련 FAQ 정보]\n'
            for r in faq_results:
                faq_context += f"Q: {r['question']}\nA: {r['answer']}\n\n"  # ← 'answer'
            return base + faq_context
        return base

    def chat(self, user_message):
        """사용자 메시지를 받아 FAQ 검색 후 응답 생성"""
        # 1단계: FAQ 검색
        faq_results   = search_faq(user_message)
        system_prompt = self._build_system_prompt(faq_results)

        # 2단계: 히스토리에 사용자 메시지 추가
        self.history.append({
            'role': 'user',                           # ← 'user'
            'content': [{'text': user_message}]             # ← user_message
        })

        # 히스토리 크기 제한 (최근 N개만 유지)
        if len(self.history) > self.max_history:
            self.history = self.history[-self.max_history:]

        # 3단계: Converse API 호출
        response = self.client.converse(
            modelId=self.model_id,                           # ← self.model_id
            messages=self.history,                          # ← self.history
            system=[{'text': system_prompt}],
            inferenceConfig={'maxTokens': 512, 'temperature': 0.3}
        )

        # 4단계: 응답 파싱 및 히스토리 추가
        answer = response['output']['message']['content'][0]['text']  # ← 'message', 'text'
        self.history.append({'role': 'assistant', 'content': [{'text': answer}]})

        return answer, faq_results

    def reset(self):
        self.history = []
        print('대화 히스토리 초기화 완료')

print('✅ FAQChatbot 클래스 정의 완료')


✅ FAQChatbot 클래스 정의 완료


## ✏️ TODO 3: 챗봇 실행 & 멀티턴 테스트

챗봇을 인스턴스화하고 연속 질문으로 멀티턴 대화를 테스트하세요.


In [6]:
# ✏️ TODO 3: FAQChatbot을 생성하고 멀티턴 대화를 테스트하세요
bot = FAQChatbot()   # ← FAQChatbot

questions = [
    '수강신청은 언제 하나요?',
    '취소는 어떻게 해요?',          # ← 이전 대화 맥락(수강신청) 이해 필요
    '성적 장학금 기준도 알려줘요',
    '졸업하려면 학점이 몇 개 필요해요?',
]

for q in questions:
    print(f'\n👤 학생: {q}')
    answer, faqs = bot.chat(q)   # ← chat
    print(f'🤖 챗봇: {answer}')
    if faqs:
        print(f'   [참조 FAQ: {faqs[0]["category"]} - {faqs[0]["question"][:25]}...]')



👤 학생: 수강신청은 언제 하나요?


🤖 챗봇: 수강신청은 매 학기 개강 2주 전에 진행됩니다. 

학년별 일정은 다음과 같습니다:
- 1학년: 3월 1일~3일
- 2학년 이상: 2월 27일~28일

미리 수강 계획을 세워두시고 해당 기간에 학생포털을 통해 신청하시기 바랍니다.
   [참조 FAQ: 수강신청 - 수강신청은 언제 하나요?...]

👤 학생: 취소는 어떻게 해요?


🤖 챗봇: 수강신청 취소는 학생포털 → 수강관리 → 수강취소 메뉴에서 가능합니다. 

개강 후 2주 이내까지 취소할 수 있으니, 필요하시면 해당 기간 내에 처리하시기 바랍니다.

더 자세한 사항이 궁금하시면 학생처(02-XXX-XXXX)에 문의해 주세요.
   [참조 FAQ: 수강신청 - 수강신청 취소는 어떻게 하나요?...]

👤 학생: 성적 장학금 기준도 알려줘요


🤖 챗봇: 성적 장학금 기준은 다음과 같습니다.

**기본 조건**: 직전 학기 12학점 이상 수강
**장학금 구분**:
- 전액 장학금: 평점 3.8 이상
- 반액 장학금: 평점 3.5~3.79

성적을 잘 관리하셔서 장학금 혜택을 받으시길 바랍니다!
   [참조 FAQ: 장학금 - 성적 장학금 기준은?...]

👤 학생: 졸업하려면 학점이 몇 개 필요해요?


🤖 챗봇: 졸업하려면 총 130학점이 필요합니다.

학점 구성은 다음과 같습니다:
- 전공 60학점
- 교양 30학점  
- 자유선택 40학점

각 영역별로 필요한 학점을 모두 충족해야 졸업이 가능하니, 학기별로 계획적으로 수강하시기 바랍니다.
   [참조 FAQ: 성적 - 졸업 학점은 몇 학점인가요?...]


## ✏️ TODO 4: 스트리밍 응답 구현

`converse_stream`을 사용하여 응답을 실시간으로 출력하는 기능을 추가하세요.


In [7]:
# ✏️ TODO 4: converse_stream으로 스트리밍 응답을 구현하세요
def chat_stream(user_message, system_prompt):
    """스트리밍으로 응답을 실시간 출력하는 함수"""
    response = bedrock_runtime.converse_stream(
        modelId=MODEL_ID,   # ← MODEL_ID
        messages=[{'role': 'user', 'content': [{'text': user_message}]}],  # ← user_message
        system=[{'text': system_prompt}],
        inferenceConfig={'maxTokens': 512, 'temperature': 0.3}
    )

    full_text = ''
    print('🤖 챗봇 (스트리밍): ', end='', flush=True)

    for event in response['stream']:  # ← 'stream'
        if 'contentBlockDelta' in event:
            delta = event['contentBlockDelta']['delta']
            if 'text'  in delta:   # ← 'text'
                chunk = delta['text']  # ← 'text'
                print(chunk, end='', flush=True)
                full_text += chunk

    print()  # 줄바꿈
    return full_text

SYSTEM = bot._build_system_prompt(search_faq('성적 이의신청'))
chat_stream('성적 이의신청은 어떻게 하나요?', SYSTEM)


🤖 챗봇 (스트리밍): 

성적 이

의신청은 

성적 발표 

후 1주일

...

인하시고 

빠르게 신

청해

주세요.

'성적 이의신청은 성적 발표 후 1주일 이내에 학생포털에서 신청하실 수 있습니다. \n\n또한 담당 교수님께 직접 문의하는 방법도 있으니, 상황에 따라 적절한 방법을 선택하시면 됩니다. \n\n기한을 놓치지 않도록 성적 발표일을 확인하시고 빠르게 신청해 주세요.'

## ✏️ TODO 5: 챗봇 품질 평가

테스트 케이스를 정의하고 챗봇 응답의 정확도를 평가하세요.


In [8]:
# ✏️ TODO 5: 챗봇 품질 평가 함수를 완성하세요
TEST_CASES = [
    {'question': '수강신청 최대 학점은?',           'expected_keywords': ['21학점', '24학점']},
    {'question': '성적 이의신청 기간은?',           'expected_keywords': ['1주일']},
    {'question': '장학금 성적 기준 알려줘',         'expected_keywords': ['3.8', '3.5']},
    {'question': '오늘 점심 메뉴 추천해줘',         'expected_keywords': ['학생처', '문의']},  # 범위 외 질문
]

def evaluate_chatbot(bot, test_cases):
    """테스트 케이스 기반 챗봇 평가"""
    results = []
    bot.reset()

    for tc in test_cases:
        answer, _ = bot.chat(tc['question'])  # ← 'question'

        # 기대 키워드 포함 여부 확인
        found = [kw for kw in tc['expected_keywords'] if kw in answer]  # ← answer
        passed = len(found) > 0  # ← 0

        results.append({
            '질문'      : tc['question'],
            '통과'      : '✅' if passed else '❌',
            '매칭키워드': str(found),
            '응답일부'  : answer[:50] + '...'
        })

    import pandas as pd
    df = pd.DataFrame(results)
    pass_rate = (df['통과'] == '✅').mean() * 100
    print(df.to_string(index=False))
    print(f'\n📊 통과율: {pass_rate:.0f}% ({int(pass_rate/100*len(test_cases))}/{len(test_cases)})')
    return df

eval_bot = FAQChatbot()
eval_df  = evaluate_chatbot(eval_bot, TEST_CASES)


대화 히스토리 초기화 완료


           질문 통과            매칭키워드                                                    응답일부
 수강신청 최대 학점은?  ✅ ['21학점', '24학점'] 수강신청 최대 학점은 일반적으로 **21학점**까지 가능합니다. \n\n다만, 직전 학기 성적...
 성적 이의신청 기간은?  ✅          ['1주일'] 성적 이의신청은 **성적 발표 후 1주일 이내**에 가능합니다.\n\n학생포털에서 온라인으로 ...
장학금 성적 기준 알려줘  ✅   ['3.8', '3.5'] 성적 장학금 기준은 다음과 같습니다.\n\n**전액 장학금**: 직전 학기 12학점 이상 수강...
오늘 점심 메뉴 추천해줘  ✅    ['학생처', '문의'] 해당 내용은 학생처(02-XXX-XXXX)에 문의해 주세요.\n\n학생처 FAQ 챗봇은 학사 ...

📊 통과율: 100% (4/4)


---
## 🔗 실무로 연결하기

오늘 만든 FAQ 챗봇이 실서비스에서는 이렇게 확장됩니다.

`사용자` → `API Gateway` → `Lambda(검색+Bedrock)` → `스트리밍 응답`

- 키워드 검색의 한계(오타·동의어에 약함) → **Part 2의 벡터 검색(Knowledge Base)** 으로 해결
- 멀티턴 히스토리 관리·스트리밍은 실제 상담 챗봇의 필수 요소
- 품질 평가 자동화로 답변 정확도를 지속 모니터링


## 💡 심화 도전
1. FAQ에 없는 질문을 받았을 때 자동으로 '자주 묻는 질문 카테고리' 목록을 보여주는 기능을 추가하세요.
2. 사용자 질문 로그를 저장하고 가장 많이 묻는 질문 Top 5를 분석하는 기능을 구현하세요.
3. Gradio 또는 ipywidgets로 간단한 챗봇 UI를 만들어보세요.


## ✅ 정답 코드

👆 모두 풀고 난 후 확인하세요

```python
# TODO 1 - search_faq
for kw in category['keywords'] if kw in query
category['category'], qa['a']
results.sort(key=lambda x: x['score'], reverse=True)

# TODO 2 - FAQChatbot.chat
faq_results   = search_faq(user_message)
system_prompt = self._build_system_prompt(faq_results)
self.history.append({'role': 'user', 'content': [{'text': user_message}]})
response = self.client.converse(modelId=self.model_id, messages=self.history, ...)
answer = response['output']['message']['content'][0]['text']

# TODO 3
bot = FAQChatbot()
answer, faqs = bot.chat(q)

# TODO 4 - chat_stream
response = bedrock_runtime.converse_stream(modelId=MODEL_ID,
    messages=[{'role': 'user', 'content': [{'text': user_message}]}], ...)
for event in response['stream']:
    if 'contentBlockDelta' in event:
        delta = event['contentBlockDelta']['delta']
        if 'text' in delta: chunk = delta['text']

# TODO 5 - evaluate_chatbot
answer, _ = bot.chat(tc['question'])
found  = [kw for kw in tc['expected_keywords'] if kw in answer]
passed = len(found) > 0
```
